# 03 — Retrieval, selection và simulated annotation

**Input:** output NB1. **GPU:** nên có (embed pool). **Output:** `art/retrieval/`, `art/selections/`.

Thứ tự bắt buộc: embed + chọn mẫu trong `no_oracle_access()` → chốt manifest → mới mở oracle.
Nhánh `ORACLE_POSITIVE` và `RETRIEVAL_MATCHED` dùng nhãn ẩn (privileged), được ghi log.

In [ ]:
# === Setup (giống nhau ở mọi notebook) ===
SMOKE = True   # True: pipeline thật trên tập nhỏ. Run thật: False
REPO_URL = "https://github.com/Le-Anh-Duy/topicx-yolo-expansion.git"
COMMIT = "main"                                    # run thật: pin commit hash
import os, subprocess, sys
R = "/tmp/repo"
if not os.path.exists(R):
    subprocess.run(["git", "clone", "-q", REPO_URL, R], check=True)
subprocess.run(["git", "-C", R, "fetch", "-q"], check=True)
subprocess.run(["git", "-C", R, "checkout", "-q", COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{R}/requirements.txt"], check=True)
sys.path.insert(0, f"{R}/src")
import numpy as np, pandas as pd
from topicx import common as C
cfg = C.load_config(SMOKE)
print(cfg["art"])

In [ ]:
STAGE = "retrieval"
# Test protocol (CPU, vài giây). Lỗi thì dừng notebook.
subprocess.run([sys.executable, "-m", "pytest", "-q", f"{R}/tests"], check=True)
C.env_report(cfg, STAGE)

In [ ]:
from topicx import data as D, embed as E, select as S, metrics as M
from topicx.oracle import OracleStore, no_oracle_access
meta = C.load(C.find(cfg, "splits/meta.json"))
NOVEL = meta["novel"]
B = cfg["branches"] + cfg["analysis_branches"]
paths = D.split_paths(cfg)
pool = C.read_ids(C.find(cfg, "splits/pool_ids.txt"))
dev = C.read_ids(C.find(cfg, "splits/dev_ids.txt"))
pub_boxes = pd.read_csv(C.find(cfg, "splits/public_boxes.csv"))

In [ ]:
# Embedding (không nhãn)
with no_oracle_access():
    m, pre, tok = E.load_clip(cfg["clip"]["model"], cfg["clip"]["pretrained"])
    clip_pool = E.clip_images([paths[i] for i in pool], m, pre, cfg["clip"]["batch"])
    clip_dev = E.clip_images([paths[i] for i in dev], m, pre, cfg["clip"]["batch"])
    dino_pool = E.dinov2_images([paths[i] for i in pool], cfg["dinov2"]["model"], cfg["dinov2"]["batch"])
np.save(C.out(cfg, "retrieval/clip_pool.npy"), clip_pool)
np.save(C.out(cfg, "retrieval/clip_dev.npy"), clip_dev)
np.save(C.out(cfg, "retrieval/dino_pool.npy"), dino_pool)
import open_clip
C.dump({"clip": cfg["clip"], "open_clip": open_clip.__version__, "preprocess": str(pre),
        "crops": "3 crop vuông trái/giữa/phải, mỗi crop qua preprocess của checkpoint", "score": "max_crop cos(img, mean text emb)",
        "dinov2": cfg["dinov2"], "dinov2_preprocess": "resize 224x392, ImageNet norm"}, C.out(cfg, "retrieval/embed_meta.json"))

In [ ]:
# Chọn prompt trên dev (nhãn dev công khai). Luật khai báo trước: AP ranking cao nhất.
prompts = cfg["topic_prompts"][NOVEL]
T = E.clip_texts(prompts, m, tok)
dev_pos = set(pub_boxes.image[pub_boxes.cls == NOVEL])
y_dev = [i in dev_pos for i in dev]
cands = {f"single:{p}": [j] for j, p in enumerate(prompts)} | {"ensemble:all": list(range(len(prompts)))}
dev_ap = {name: M.average_precision(E.topic_scores(clip_dev, T[idx]), y_dev) for name, idx in cands.items()}
best = max(dev_ap, key=dev_ap.get)
chosen = cands[best]
C.dump({"rule": "max dev ranking AP", "dev_ap": dev_ap, "best": best, "prompts": [prompts[j] for j in chosen],
        "dev_positive_rate": float(np.mean(y_dev))}, C.out(cfg, "retrieval/prompt_selection.json"))
pd.Series(dev_ap).sort_values()

In [ ]:
# Selection không nhãn -> chốt manifest
SEL = lambda s, k, b: C.out(cfg, f"selections/s{s}_k{k}_{b}.json")
md = cfg["diverse"]["min_dist"]
with no_oracle_access():
    scores = E.topic_scores(clip_pool, T[chosen])
    order = S.rank(pool, scores)
    clip_img = E.image_level(clip_pool)
    for s in cfg["seeds"]:
        ss = 1000 + s
        for k in cfg["ks"]:
            if "RANDOM" in B:
                C.freeze(SEL(s, k, "RANDOM"), S.random_k(pool, k, ss), branch="RANDOM", seed=s, k=k)
            if "DIVERSITY" in B:
                perm = np.random.default_rng(ss).permutation(len(pool))
                ids, f = S.greedy_diverse(pool, perm, dino_pool, k, md)
                C.freeze(SEL(s, k, "DIVERSITY"), ids, branch="DIVERSITY", seed=s, k=k, relax=f, emb="dinov2", min_dist=md)
            if "RETRIEVAL" in B:
                C.freeze(SEL(s, k, "RETRIEVAL"), [pool[i] for i in order[:k]], branch="RETRIEVAL", seed=s, k=k)
            if "RETRIEVAL_DIVERSE" in B:
                ids, f = S.greedy_diverse(pool, order[:cfg["diverse"]["max_rank_mult"] * k], clip_img, k, md)
                C.freeze(SEL(s, k, "RETRIEVAL_DIVERSE"), ids, branch="RETRIEVAL_DIVERSE", seed=s, k=k, relax=f, emb="clip", min_dist=md)
np.save(C.out(cfg, "retrieval/pool_scores.npy"), scores)

In [ ]:
# Từ đây mới mở oracle
oracle = OracleStore(C.find(cfg, "splits/oracle"), C.out(cfg, "retrieval/oracle_log.jsonl"))
pos = oracle.privileged_positive_ids(NOVEL, reason="ORACLE_POSITIVE branch + recall@K denominator")
if "ORACLE_POSITIVE" in B:
    neg = sorted(set(pool) - set(pos))
    for s in cfg["seeds"]:
        for k in cfg["ks"]:
            ids = S.random_k(pos, k, 1000 + s)
            pad = k - len(ids)
            ids += S.random_k(neg, pad, 2000 + s) if pad else []
            C.freeze(SEL(s, k, "ORACLE_POSITIVE"), ids, branch="ORACLE_POSITIVE", seed=s, k=k, privileged=True, padded_negatives=pad)

In [ ]:
# Simulated annotation + đánh giá retrieval
dix = {i: j for j, i in enumerate(pool)}
rows = []
for s in cfg["seeds"]:
    for k in cfg["ks"]:
        for b in [b for b in B if b not in ("REPLAY_ONLY", "RETRIEVAL_MATCHED")]:
            bx, im = oracle.reveal(SEL(s, k, b))
            ids = C.read_frozen(SEL(s, k, b))["ids"]
            rows.append({"seed": s, "k": k, "branch": b, **M.retrieval_report(ids, bx, im, NOVEL, len(pos)),
                         **M.batch_redundancy(dino_pool[[dix[i] for i in ids]])})
ret = pd.DataFrame(rows).fillna(0)
ret.to_csv(C.out(cfg, "retrieval/retrieval.csv"), index=False)
display(ret.groupby(["k", "branch"]).mean(numeric_only=True).drop(columns="seed"))

In [ ]:
# (Phân tích, privileged) RETRIEVAL_MATCHED: prefix ngắn nhất của ranking có >= số novel instance của RANDOM cùng (seed, K);
# phần còn lại bù bằng ảnh control từ base train để giữ đủ K ảnh (cùng số optimizer update).
if "RETRIEVAL_MATCHED" in B:
    control = C.read_ids(C.find(cfg, "splits/control_ids.txt"))
    ranked = [pool[i] for i in order]
    for s in cfg["seeds"]:
        for k in cfg["ks"]:
            target = int(ret.query("seed == @s and k == @k and branch == 'RANDOM'").novel_instances.iloc[0])
            cum = oracle.privileged_instance_counts(ranked[:k], NOVEL, reason="RETRIEVAL_MATCHED").cumsum().to_numpy()
            kp = min(int(np.searchsorted(cum, target)) + 1, k) if target else 0
            C.freeze(SEL(s, k, "RETRIEVAL_MATCHED"), ranked[:kp], branch="RETRIEVAL_MATCHED", seed=s, k=k, privileged=True,
                     pad_ids=control[:k - kp], target_novel_instances=target, got_novel_instances=int(cum[kp - 1]) if kp else 0)